# Fleet: interactive walkthrough

Many **agents** (weather stations) each call **out** to one **control plane**: register once, then heartbeat forever.
The plane keeps the registry, infers health from silence, and hands back desired state in every reply.

The notebook must live next to the `fleet/` package (it imports `fleet` by name). Launch with:

```bash
uv run --with jupyterlab jupyter lab fleet.ipynb
```

Stations run as background asyncio tasks, so they keep heartbeating *between* cells. Agent log lines go to `events`; `show()` prints the new ones, then the dashboard.

In [1]:
import asyncio
import random
import tempfile
from functools import partial
from pathlib import Path

from fleet import Backoff, ControlPlane
from fleet import demo

random.seed(7)
events: list[str] = []        # agent log lines land here instead of stray cell outputs

# The demo's composition root, with agent logs routed into `events`.
start_station = partial(demo.start_station, say=events.append)


async def show(plane: ControlPlane, seconds: float = 1.5) -> None:
    """Let the fleet run, print what the agents said, then the operator's dashboard."""
    await asyncio.sleep(seconds)
    print("\n".join(events) or "  (no agent events)")
    events.clear()
    print()
    plane.print_dashboard()

## 1. Startup: register, then heartbeat

Each station gets a stable id, registers (an idempotent upsert), then heartbeats every 0.3 s.
The plane never calls anyone: it only answers. A few `503 → backoff` lines are the simulated flaky network.

In [2]:
plane = ControlPlane(latest_firmware="2.0.0")
id_dir = Path(tempfile.mkdtemp())                 # each station's "disk"
stations = {n: start_station(n, "2.0.0", plane, id_dir) for n in ("rooftop-A", "hilltop-B", "valley-C")}
await show(plane)

  [rooftop-A] registered as 3218f900
  [hilltop-B] registered as f6ccc278
  [valley-C] registered as bcf217d4
  [valley-C] 503 → backoff 0.02s

  NAME        ID        FIRMWARE  LINK    WORKLOAD  TEMP°C
  hilltop-B   f6ccc278  2.0.0     online  alive     27.5
  rooftop-A   3218f900  2.0.0     online  alive     20.9
  valley-C    bcf217d4  2.0.0     online  alive     24.0


The dashboard is plain data underneath: one row per registry record. Everything in it came *from* the stations.

In [3]:
plane.dashboard()

[{'name': 'hilltop-B',
  'id': 'f6ccc278',
  'firmware': '2.0.0',
  'link': 'online',
  'workload': 'alive',
  'temp_c': 27.5},
 {'name': 'rooftop-A',
  'id': '3218f900',
  'firmware': '2.0.0',
  'link': 'online',
  'workload': 'alive',
  'temp_c': 20.9},
 {'name': 'valley-C',
  'id': 'bcf217d4',
  'firmware': '2.0.0',
  'link': 'online',
  'workload': 'alive',
  'temp_c': 24.0}]

## 2. A station dies silently

Nobody tells the plane that valley-C lost power. The plane notices only because the heartbeats **stop**: `link` flips to `STALE` once `last_seen` is older than `stale_after`.
The other columns are the *last known* values: a stale row's `alive` is stale too.

In [4]:
for t in stations["valley-C"][1]:
    t.cancel()
await show(plane)

  (no agent events)

  NAME        ID        FIRMWARE  LINK    WORKLOAD  TEMP°C
  hilltop-B   f6ccc278  2.0.0     online  alive     24.4
  rooftop-A   3218f900  2.0.0     online  alive     26.1
  valley-C    bcf217d4  2.0.0     STALE   alive     24.0


## 3. Reboot keeps the identity

valley-C restarts and reads the same id file, so it re-registers under the **same** `station_id`: same row, no ghost duplicate.
Try deleting `id_dir / "valley-C.id"` before this cell: a second valley-C row appears, and the old one stays STALE forever.

In [5]:
stations["valley-C"] = start_station("valley-C", "2.0.0", plane, id_dir)
await show(plane)
len(plane.registry)

  [valley-C] registered as bcf217d4
  [valley-C] 503 → backoff 0.02s
  [hilltop-B] 503 → backoff 0.02s

  NAME        ID        FIRMWARE  LINK    WORKLOAD  TEMP°C
  hilltop-B   f6ccc278  2.0.0     online  alive     21.3
  rooftop-A   3218f900  2.0.0     online  alive     21.2
  valley-C    bcf217d4  2.0.0     online  alive     24.8


3

## 4. Link health ≠ work health

hilltop-B's sensor hangs. Its agent still heartbeats (`link: online`), but reports `workload_alive: False`, and its temperature stops changing.
A heartbeat that only proves "the process is up" would miss this.

In [6]:
stations["hilltop-B"][0].frozen = True
await show(plane)

  [rooftop-A] 503 → backoff 0.04s

  NAME        ID        FIRMWARE  LINK    WORKLOAD  TEMP°C
  hilltop-B   f6ccc278  2.0.0     online  DEAD      31.3
  rooftop-A   3218f900  2.0.0     online  alive     21.5
  valley-C    bcf217d4  2.0.0     online  alive     22.9


## 5. Self-healing after a control-plane restart

The plane loses its registry. It does not need a backup of "who exists": the next heartbeat from each station gets a `401 unknown station`, the station re-registers, and the registry rebuilds itself.
**The stations are the source of truth for their own existence.**

In [7]:
plane.forget_all()
plane.print_dashboard()          # empty: the plane knows nobody

  NAME        ID        FIRMWARE  LINK    WORKLOAD  TEMP°C


In [8]:
await show(plane)                # rebuilt from 401 → re-register

  [valley-C] 401 unknown station → re-register
  [valley-C] registered as bcf217d4
  [hilltop-B] 401 unknown station → re-register
  [hilltop-B] registered as f6ccc278
  [rooftop-A] 503 → backoff 0.05s
  [rooftop-A] 401 unknown station → re-register
  [rooftop-A] registered as 3218f900
  [rooftop-A] 503 → backoff 0.03s

  NAME        ID        FIRMWARE  LINK    WORKLOAD  TEMP°C
  hilltop-B   f6ccc278  2.0.0     online  DEAD      31.3
  rooftop-A   3218f900  2.0.0     online  alive     28.4
  valley-C    bcf217d4  2.0.0     online  alive     18.4


## 6. Desired state rides back on the reply

The operator changes **one value on the plane**. No push, no list of addresses to call: every heartbeat reply already carries `latest_firmware`, and each agent reconciles toward it.
`*` marks a row whose reported firmware differs from desired.

In [9]:
plane.latest_firmware = "2.1.0"
plane.print_dashboard()          # * = actual ≠ desired, until the next heartbeats

  NAME        ID        FIRMWARE  LINK    WORKLOAD  TEMP°C
  hilltop-B   f6ccc278  2.0.0*    online  DEAD      31.3
  rooftop-A   3218f900  2.0.0*    online  alive     28.4
  valley-C    bcf217d4  2.0.0*    online  alive     18.4


In [10]:
await show(plane)                # converged

  [rooftop-A] firmware 2.0.0 → 2.1.0 (stub flash)
  [valley-C] firmware 2.0.0 → 2.1.0 (stub flash)
  [hilltop-B] firmware 2.0.0 → 2.1.0 (stub flash)

  NAME        ID        FIRMWARE  LINK    WORKLOAD  TEMP°C
  hilltop-B   f6ccc278  2.1.0     online  DEAD      31.3
  rooftop-A   3218f900  2.1.0     online  alive     29.0
  valley-C    bcf217d4  2.1.0     online  alive     22.7


## 7. Why jittered backoff

After an outage, every station retries at once. Exponential backoff spreads retries **out in time**; full jitter (a random delay in `[0, window]`) keeps them from falling back into lock-step.

In [11]:
b = Backoff()
for attempt in range(7):
    window = min(b.cap, b.base * 2**attempt)
    samples = sorted(round(random.uniform(0, window), 2) for _ in range(5))
    print(f"attempt {attempt}: window {window:.2f}s   5 stations retry at {samples}")

attempt 0: window 0.05s   5 stations retry at [0.01, 0.01, 0.01, 0.04, 0.05]
attempt 1: window 0.10s   5 stations retry at [0.01, 0.08, 0.08, 0.09, 0.1]
attempt 2: window 0.20s   5 stations retry at [0.0, 0.03, 0.07, 0.11, 0.13]
attempt 3: window 0.40s   5 stations retry at [0.17, 0.21, 0.26, 0.37, 0.39]
attempt 4: window 0.80s   5 stations retry at [0.17, 0.2, 0.23, 0.66, 0.7]
attempt 5: window 1.00s   5 stations retry at [0.13, 0.24, 0.26, 0.42, 0.59]
attempt 6: window 1.00s   5 stations retry at [0.35, 0.46, 0.58, 0.9, 0.91]


## Clean up

Stop every background task, or they keep running in this kernel.

In [12]:
for _, tasks in stations.values():
    for t in tasks:
        t.cancel()